# المشروع 2 — تتبّع من درون (كاميرا متحركة)
**YOLO11 + BoT-SORT + تعويض حركة الكاميرا (Camera Motion Compensation)**

## لماذا لا يصلح المشروع 1 لفيديو الدرون؟
المشروع 1 افترض **كاميرا ثابتة**: خط العدّ والمنطقة والسرعة كلها مرسومة على إحداثيات الصورة. مع الدرون تتحرك الصورة كلها، فيحدث الآتي:
- **الخط والمنطقة الثابتان بلا معنى:** المركبة قد «تعبر الخط» لأن الكاميرا تحركت لا لأن المركبة تحركت. لذلك **حُذفا** من هذا المشروع.
- **السرعة والاتجاه خاطئان:** سيارة واقفة تبدو متحركة، وسيارة متحركة قد تبدو واقفة.
- **التوقف والسير العكسي** يعتمدان على الحركة الحقيقية، فلا يمكن حسابهما من إحداثيات الصورة.

## الحل: تعويض حركة الكاميرا
```
كل إطار ─► YOLO + BoT-SORT ─► صناديق + IDs
   │
   └─► ميزات الخلفية (بدون صناديق المركبات وحواف الصورة)
        ─► Optical Flow (Lucas-Kanade)
        ─► Homography + RANSAC (إزالة القيم الشاذة)
        ─► تراكم التحويلات ─► H : إطار الآن → إطار مرجعي (الإطار الأول)
                                   │
   نقطة تماس المركبة مع الأرض ────► إحداثيات «أرضية» مستقرة ─► سرعة / اتجاه / توقف
```
- المسارات تُخزَّن بالإحداثيات المستقرة ثم تُرسم على كل إطار بالتحويل العكسي، فتبقى **ملتصقة بالأرض** ولا تنزلق مع حركة الكاميرا.
- BoT-SORT يحتوي تعويضًا داخليًا للحركة (GMC) لتحسين **ربط الـ IDs** فقط، أما هنا فنحسب التحويل بأنفسنا لنستعمله في **التحليل**.
- جدول النتائج يعرض **Raw image speed** بجانب **Ground speed** لترى بنفسك أثر الكاميرا.

## 1. التثبيت

In [ ]:
!pip install -q ultralytics opencv-python pandas matplotlib lap

## 2. الإعدادات

In [13]:
VIDEO      = r"C:\Users\PC-LAB1\Desktop\New folder\day4\british_highway_traffic.mp4"          # path to your drone clip
OUT_VIDEO  = "drone_tracked.mp4"
OUT_CSV    = "drone_report.csv"

MODEL      = "yolo11s.pt"               # small objects from a drone: use s/m rather than n
TRACKER    = "botsort.yaml"             # includes its own camera-motion compensation for ID association
CLASSES    = [0, 2, 3, 5, 7]            # COCO: person, car, motorcycle, bus, truck
CONF, IMGSZ = 0.25, 960                 # larger imgsz helps tiny people/cars

# None -> speeds are reported in px/s (reference-frame pixels).
# Set meters-per-pixel only if you can calibrate it (e.g. a known object length near the ground plane).
M_PER_PX   = None
MOVE_THR   = None                       # None -> 12 px/s (or 3 km/h when calibrated); raise it if static cars look "moving"
STOP_SECS  = 2.0

MAX_FRAMES = None                       # e.g. 300 for a quick test
PREVIEW_FRAME = 120

## 3. وحدة تعويض الحركة والتحليل
اختُبرت على فيديو اصطناعي تتحرك كاميرته: سيارة ثابتة ظهرت بسرعة صورة ≈ 119 px/s وبسرعة أرضية ≈ 0.7 px/s (Static)، وسيارة تتحرك بسرعة فعلية 60 px/s قُدِّرت 59.4.

In [14]:
import math
from dataclasses import dataclass, field
import cv2
import numpy as np
import pandas as pd

def compass(dx, dy, min_move=5.0):
    """Reference-frame directions. Image 'up' in the first frame = North."""
    if math.hypot(dx, dy) < min_move:
        return "-"
    ang = math.degrees(math.atan2(-dy, dx))
    names = ["East", "North-East", "North", "North-West",
             "West", "South-West", "South", "South-East"]
    return names[int(((ang + 22.5) % 360) // 45)]

class CameraMotion:
    """Estimates frame-to-frame camera motion (homography from tracked background
    features) and accumulates it so every frame can be mapped to the first frame."""
    def __init__(self, max_corners=500, ransac_px=3.0, border=0.08, min_inliers=12):
        self.max_corners, self.ransac_px = max_corners, ransac_px
        self.border, self.min_inliers = border, min_inliers
        self.H = np.eye(3)                       # current frame -> reference frame
        self.prev_gray = self.prev_mask = None
        self.inlier_ratio, self.failures = 1.0, 0

    def update(self, frame, boxes=()):
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        h, w = gray.shape
        mask = np.full((h, w), 255, np.uint8)
        by, bx = int(h * self.border), int(w * self.border)
        mask[:by] = 0; mask[h - by:] = 0         # skip borders (OSD text, vignetting)
        mask[:, :bx] = 0; mask[:, w - bx:] = 0
        for x1, y1, x2, y2 in boxes:             # skip moving objects: they are not background
            mask[max(int(y1) - 10, 0):int(y2) + 10, max(int(x1) - 10, 0):int(x2) + 10] = 0

        if self.prev_gray is not None:
            ok = False
            p0 = cv2.goodFeaturesToTrack(self.prev_gray, self.max_corners, 0.01, 8, mask=self.prev_mask)
            if p0 is not None and len(p0) >= 8:
                p1, st, _ = cv2.calcOpticalFlowPyrLK(self.prev_gray, gray, p0, None)
                good = st.ravel() == 1
                a, b = p0[good], p1[good]
                if len(a) >= 8:
                    Hrel, inl = cv2.findHomography(b, a, cv2.RANSAC, self.ransac_px)   # current -> previous
                    if Hrel is not None and int(inl.sum()) >= self.min_inliers:
                        self.H = self.H @ Hrel
                        self.inlier_ratio = float(inl.mean())
                        ok = True
            if not ok:
                self.failures += 1               # keep last transform (identity step)
        self.prev_gray, self.prev_mask = gray, mask
        return self.H

    def to_ref(self, pt):
        v = self.H @ np.array([pt[0], pt[1], 1.0])
        return (float(v[0] / v[2]), float(v[1] / v[2]))

    def ref_to_frame(self, pts):
        Hi = np.linalg.inv(self.H).astype(np.float32)
        arr = np.asarray(pts, np.float32).reshape(-1, 1, 2)
        return cv2.perspectiveTransform(arr, Hi).reshape(-1, 2)

@dataclass
class Track:
    cls: str
    raw: list = field(default_factory=list)      # (frame, x, y) in the image
    ref: list = field(default_factory=list)      # (frame, x, y) camera-motion compensated
    still_since: int = None
    ever_moved: bool = False
    stopped: bool = False

class DroneAnalytics:
    def __init__(self, fps, m_per_px=None, win_s=1.0, move_thr=None, stop_secs=2.0):
        self.fps, self.win, self.stop_secs = fps, win_s, stop_secs
        if m_per_px:
            self.k, self.unit, self.move_thr = m_per_px * 3.6, "km/h", move_thr or 3.0
        else:
            self.k, self.unit, self.move_thr = 1.0, "px/s", move_thr or 12.0
        self.tracks = {}

    def _speed(self, pts, i):
        f_now = pts[i][0]
        j = i
        while j > 0 and f_now - pts[j - 1][0] <= self.win * self.fps:
            j -= 1
        dt = (pts[i][0] - pts[j][0]) / self.fps
        if dt <= 0:
            return 0.0
        return math.hypot(pts[i][1] - pts[j][1], pts[i][2] - pts[j][2]) / dt * self.k

    def update(self, frame_no, tid, cls_name, raw_pt, ref_pt):
        t = self.tracks.setdefault(tid, Track(cls_name))
        t.raw.append((frame_no, *raw_pt))
        t.ref.append((frame_no, *ref_pt))
        v = self._speed(t.ref, len(t.ref) - 1)
        age = (t.ref[-1][0] - t.ref[0][0]) / self.fps
        if age >= self.win:
            if v >= self.move_thr:
                t.ever_moved, t.still_since, t.stopped = True, None, False
            elif t.ever_moved:
                if t.still_since is None:
                    t.still_since = frame_no
                if (frame_no - t.still_since) / self.fps >= self.stop_secs:
                    t.stopped = True
        return v

    def unique_counts(self, min_frames=15):
        out = {}
        for t in self.tracks.values():
            if len(t.ref) >= min_frames:
                out[t.cls] = out.get(t.cls, 0) + 1
        return out

    def report(self, min_frames=15):
        rows = []
        for tid, t in self.tracks.items():
            if len(t.ref) < min_frames:
                continue
            g = [self._speed(t.ref, i) for i in range(len(t.ref))]
            r = [self._speed(t.raw, i) for i in range(len(t.raw))]
            dx, dy = t.ref[-1][1] - t.ref[0][1], t.ref[-1][2] - t.ref[0][2]
            rows.append({
                "ID": tid, "Type": t.cls,
                "Status": "Moving" if np.mean(g) >= self.move_thr else "Static",
                f"Ground speed ({self.unit})": round(float(np.mean(g)), 1),
                "Raw image speed (px/s)": round(float(np.mean(r)) / self.k, 1),
                "Direction (ref. frame)": compass(dx, dy),
                "Seen (s)": round((t.ref[-1][0] - t.ref[0][0]) / self.fps, 1),
                "Stopped": t.stopped,
            })
        return pd.DataFrame(rows)

## 4. فتح الفيديو

In [15]:
import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

cap = cv2.VideoCapture(VIDEO)
assert cap.isOpened(), f"Cannot open video: {VIDEO}"
fps = cap.get(cv2.CAP_PROP_FPS) or 25
W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"{W}x{H} @ {fps:.1f} fps")

1364x768 @ 25.0 fps


## 5. المعالجة: Detect → Track → Compensate → Analyze → Annotate

In [ ]:
model = YOLO(MODEL)
cam = CameraMotion()
an = DroneAnalytics(fps, M_PER_PX, move_thr=MOVE_THR, stop_secs=STOP_SECS)
writer = cv2.VideoWriter(OUT_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

GREEN, ORANGE, RED = (0, 200, 0), (0, 165, 255), (0, 0, 255)
frame_no, sample, cam_path = 0, None, []
while True:
    ok, frame = cap.read()
    if not ok or (MAX_FRAMES and frame_no >= MAX_FRAMES):
        break
    frame_no += 1

    r = model.track(frame, persist=True, tracker=TRACKER, classes=CLASSES,
                    conf=CONF, imgsz=IMGSZ, verbose=False)[0]
    dets = []
    if r.boxes.id is not None:
        ids   = r.boxes.id.int().cpu().tolist()
        boxes = r.boxes.xyxy.cpu().numpy()
        clss  = r.boxes.cls.int().cpu().tolist()
        dets = list(zip(ids, boxes.tolist(), clss))

    # 1) camera motion for this frame (background features only, object boxes masked out)
    cam.update(frame, [b for _, b, _ in dets])
    cam_path.append((cam.H[0, 2], cam.H[1, 2]))

    # 2) per-object analysis in stabilized coordinates
    for tid, (x1, y1, x2, y2), c in dets:
        name = r.names[c]
        raw_pt = ((x1 + x2) / 2, y2)                 # ground-contact point in the image
        v = an.update(frame_no, tid, name, raw_pt, cam.to_ref(raw_pt))
        t = an.tracks[tid]
        color = RED if t.stopped else (GREEN if v >= an.move_thr else ORANGE)

        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
        label = f"#{tid} {name} {v:.0f}{an.unit}" + (" STOPPED" if t.stopped else "")
        cv2.putText(frame, label, (int(x1), max(int(y1) - 6, 14)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        if len(t.ref) > 1:                            # trail lives in ground coordinates -> warp back to this frame
            trail = cam.ref_to_frame([(p[1], p[2]) for p in t.ref[-40:]]).astype(np.int32)
            cv2.polylines(frame, [trail], False, color, 2)

    hud = [f"Frame: {frame_no}   camera-fit inliers: {cam.inlier_ratio:.2f}",
           "Unique: " + "  ".join(f"{k}: {n}" for k, n in an.unique_counts().items())]
    for i, line in enumerate(hud):
        cv2.putText(frame, line, (10, 25 + 24 * i), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    writer.write(frame)
    if frame_no == PREVIEW_FRAME:
        sample = frame.copy()

cap.release(); writer.release()
print(f"Done: {frame_no} frames -> {OUT_VIDEO} | camera-fit failures: {cam.failures}")

## 6. النتائج
- **Status:** Moving / Static بحسب **الحركة الأرضية** بعد تعويض الكاميرا.
- **Raw image speed:** السرعة كما تبدو في الصورة (تتأثر بحركة الكاميرا). إذا كانت كبيرة والـ Ground speed قريبة من الصفر فهذه مركبة **واقفة** والكاميرا هي المتحركة.
- **Stopped:** مركبة كانت تتحرك ثم بقيت ساكنة `STOP_SECS` ثانية أو أكثر.

In [12]:
df = an.report()
df.to_csv(OUT_CSV, index=False)
display(df.sort_values("ID").reset_index(drop=True))
print("Unique objects:", an.unique_counts())
for _, row in df[df["Stopped"]].iterrows():
    print(f"ALERT: #{row['ID']} ({row['Type']}) stopped")

fig, ax = plt.subplots(1, 2 if sample is not None else 1, figsize=(14, 4.5))
ax = np.atleast_1d(ax)
cp = np.array(cam_path)
ax[0].plot(cp[:, 0], cp[:, 1]); ax[0].set_title("Estimated camera path (ref. frame px)"); ax[0].invert_yaxis(); ax[0].set_aspect("equal")
if sample is not None:
    ax[1].imshow(cv2.cvtColor(sample, cv2.COLOR_BGR2RGB)); ax[1].axis("off"); ax[1].set_title(f"Frame {PREVIEW_FRAME}")
plt.show()

KeyError: 'ID'

In [3]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import csv
import time


# ============================================================
# 1 - PATHS
# ============================================================

base_folder = r"C:\Users\PC-LAB1\Desktop\New folder\test_vedio\VID00051.mp4"

dhaka_video = os.path.join(base_folder, "dhaka_traffic.mp4")
british_video = os.path.join(base_folder, "british_highway_traffic.mp4")

results_folder = os.path.join(base_folder, "results")
graphs_folder = os.path.join(results_folder, "graphs")

os.makedirs(results_folder, exist_ok=True)
os.makedirs(graphs_folder, exist_ok=True)


# ============================================================
# 2 - PARAMETERS
# ============================================================

MIN_AREA = 500

# Aspect ratio (width/height) range for a bounding box to be
# accepted as a "vehicle-like" object. Filters out thin/tall
# shapes such as pedestrians, poles, or noise blobs.
MIN_ASPECT_RATIO = 0.3
MAX_ASPECT_RATIO = 4.0

KERNEL_SIZE = 5

# MOG2 foreground mask values: ~127 = shadow, ~255 = real motion.
# Threshold at 200 keeps only strong-motion pixels and removes
# shadow pixels detected by detectShadows=True.
THRESHOLD = 200

HISTORY = 500

VAR_THRESHOLD = 50

DETECT_SHADOWS = True

# Set to True only when running locally with a display attached.
# On headless / server environments this MUST stay False, or
# cv2.imshow will raise an error (or hang).
SHOW_PREVIEW = False

# Window size (in frames) used to smooth the processing-FPS curve
# in Graph 2, since the raw per-frame value can be very noisy.
FPS_SMOOTHING_WINDOW = 15


# ============================================================
# 3 - FUNCTION TO GET VIDEO INFORMATION
# ============================================================

def get_video_info(video_path):

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Error: Cannot open video:", video_path)
        return None

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = frames / fps if fps > 0 else 0

    codec_value = int(cap.get(cv2.CAP_PROP_FOURCC))
    codec = "".join([
        chr(codec_value & 255),
        chr((codec_value >> 8) & 255),
        chr((codec_value >> 16) & 255),
        chr((codec_value >> 24) & 255)
    ])

    cap.release()

    return width, height, fps, frames, duration, codec


# ============================================================
# 4 - VIDEO INFORMATION
# ============================================================

print("\n======================================")
print("VIDEO INFORMATION")
print("======================================")

dhaka_info = get_video_info(dhaka_video)
british_info = get_video_info(british_video)

if dhaka_info is not None:
    print("\nDhaka Video")
    print("Width:", dhaka_info[0])
    print("Height:", dhaka_info[1])
    print("FPS:", dhaka_info[2])
    print("Frames:", dhaka_info[3])
    print("Duration:", round(dhaka_info[4], 2), "seconds")
    print("Codec:", dhaka_info[5])

if british_info is not None:
    print("\nBritish Highway Video")
    print("Width:", british_info[0])
    print("Height:", british_info[1])
    print("FPS:", british_info[2])
    print("Frames:", british_info[3])
    print("Duration:", round(british_info[4], 2), "seconds")
    print("Codec:", british_info[5])


# ============================================================
# 5 - HELPER: MOVING AVERAGE
# ============================================================

def moving_average(values, window):
    if len(values) == 0:
        return []
    values = np.array(values, dtype=float)
    kernel = np.ones(window) / window
    # 'same' keeps output length equal to input length
    smoothed = np.convolve(values, kernel, mode="same")
    return smoothed.tolist()


# ============================================================
# 6 - PROCESS VIDEOS
# ============================================================

def process_video(video_path, video_name):

    print("\n======================================")
    print("PROCESSING:", video_name)
    print("======================================")

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Error: Cannot open:", video_path)
        return None

    # --------------------------------------------------------
    # Video Information
    # --------------------------------------------------------

    original_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    original_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    video_fps = cap.get(cv2.CAP_PROP_FPS)

    if original_width == 0 or original_height == 0:
        print("Error: Invalid video dimensions for:", video_path)
        cap.release()
        return None

    # --------------------------------------------------------
    # Resize
    # --------------------------------------------------------

    width = 960
    height = int(original_height * width / original_width)

    # --------------------------------------------------------
    # Background Subtractor
    # --------------------------------------------------------

    background = cv2.createBackgroundSubtractorMOG2(
        history=HISTORY,
        varThreshold=VAR_THRESHOLD,
        detectShadows=DETECT_SHADOWS
    )

    # --------------------------------------------------------
    # Morphology Kernel
    # --------------------------------------------------------

    kernel = np.ones((KERNEL_SIZE, KERNEL_SIZE), np.uint8)

    # --------------------------------------------------------
    # Output Video (with codec fallback)
    # --------------------------------------------------------

    output_path = os.path.join(results_folder, video_name + "_processed.mp4")

    out = None
    for codec_name in ("mp4v", "avc1"):
        fourcc = cv2.VideoWriter_fourcc(*codec_name)
        candidate = cv2.VideoWriter(output_path, fourcc, video_fps, (width, height))
        if candidate.isOpened():
            out = candidate
            break

    if out is None:
        # Final fallback: AVI with XVID, which is very broadly supported.
        output_path = os.path.join(results_folder, video_name + "_processed.avi")
        fourcc = cv2.VideoWriter_fourcc(*"XVID")
        out = cv2.VideoWriter(output_path, fourcc, video_fps, (width, height))

    # --------------------------------------------------------
    # CSV File
    # --------------------------------------------------------

    csv_path = os.path.join(results_folder, video_name + "_object_counts.csv")

    csv_file = open(csv_path, "w", newline="")
    csv_writer = csv.writer(csv_file)

    csv_writer.writerow([
        "Frame",
        "Time",
        "Object Count",
        "Processing Time (ms)",
        "Processing FPS"
    ])

    # --------------------------------------------------------
    # Variables
    # --------------------------------------------------------

    frame_number = 0

    object_counts = []
    processing_times = []
    processing_fps_values = []
    time_values = []

    max_objects = 0
    min_objects = None  # set on first valid frame, avoids fake 999999 default

    # ========================================================
    # MAIN LOOP
    # ========================================================

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frame_number += 1

        # ----------------------------------------------------
        # Resize
        # ----------------------------------------------------

        frame = cv2.resize(frame, (width, height))
        original = frame.copy()

        # ----------------------------------------------------
        # Start Processing Timer
        # ----------------------------------------------------

        start_time = time.time()

        # ----------------------------------------------------
        # Background Subtraction
        # ----------------------------------------------------

        foreground = background.apply(frame)

        # ----------------------------------------------------
        # Threshold
        # ----------------------------------------------------

        _, binary = cv2.threshold(foreground, THRESHOLD, 255, cv2.THRESH_BINARY)

        # ----------------------------------------------------
        # Morphological Cleaning
        # ----------------------------------------------------

        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
        binary = cv2.dilate(binary, kernel, iterations=2)

        # ----------------------------------------------------
        # Find Contours
        # ----------------------------------------------------

        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # ----------------------------------------------------
        # Detection Image
        # ----------------------------------------------------

        detected = frame.copy()
        object_count = 0

        # ----------------------------------------------------
        # Process Objects
        # ----------------------------------------------------

        for contour in contours:

            area = cv2.contourArea(contour)

            if area < MIN_AREA:
                continue

            x, y, w, h = cv2.boundingRect(contour)

            # --- Aspect ratio filter: reject non-vehicle-like blobs ---
            if h == 0:
                continue
            aspect_ratio = w / float(h)
            if aspect_ratio < MIN_ASPECT_RATIO or aspect_ratio > MAX_ASPECT_RATIO:
                continue

            object_count += 1

            cv2.rectangle(detected, (x, y), (x + w, y + h), (0, 255, 0), 2)

            cv2.putText(
                detected,
                "Object {}".format(object_count),
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 255, 0),
                2
            )

            cv2.putText(
                detected,
                "Area: {}".format(int(area)),
                (x, y + h + 20),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 255, 255),
                2
            )

        # ----------------------------------------------------
        # End Processing Timer
        # ----------------------------------------------------

        end_time = time.time()
        processing_time = (end_time - start_time) * 1000
        processing_fps = (1000 / processing_time) if processing_time > 0 else 0

        # ----------------------------------------------------
        # Save Measurements
        # ----------------------------------------------------

        current_time = (frame_number / video_fps) if video_fps > 0 else 0

        object_counts.append(object_count)
        processing_times.append(processing_time)
        processing_fps_values.append(processing_fps)
        time_values.append(current_time)

        # ----------------------------------------------------
        # Maximum / Minimum
        # ----------------------------------------------------

        if object_count > max_objects:
            max_objects = object_count

        if min_objects is None or object_count < min_objects:
            min_objects = object_count

        # ----------------------------------------------------
        # CSV
        # ----------------------------------------------------

        csv_writer.writerow([
            frame_number,
            round(current_time, 3),
            object_count,
            round(processing_time, 3),
            round(processing_fps, 2)
        ])

        # ====================================================
        # DISPLAY INFORMATION (drawn on saved output video)
        # ====================================================

        cv2.putText(detected, "TRAFFIC ANALYSIS", (20, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        cv2.putText(detected, "Video FPS: {:.2f}".format(video_fps), (20, 70),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 0, 0), 2)

        cv2.putText(detected, "Processing FPS: {:.2f}".format(processing_fps), (20, 100),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2)

        cv2.putText(detected, "Processing Time: {:.2f} ms".format(processing_time), (20, 130),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 255), 2)

        cv2.putText(detected, "Frame: {}".format(frame_number), (20, 160),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)

        cv2.putText(detected, "Moving Objects: {}".format(object_count), (20, 190),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 0), 2)

        # ----------------------------------------------------
        # Save Processed Video
        # ----------------------------------------------------

        out.write(detected)

        # ====================================================
        # OPTIONAL LIVE PREVIEW (local runs only)
        # ====================================================

        if SHOW_PREVIEW:

            display_width = 480
            display_height = 300

            original_small = cv2.resize(original, (display_width, display_height))
            mask_small = cv2.resize(binary, (display_width, display_height))
            mask_small = cv2.cvtColor(mask_small, cv2.COLOR_GRAY2BGR)
            detected_small = cv2.resize(detected, (display_width, display_height))

            cv2.putText(original_small, "Original", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
            cv2.putText(mask_small, "Foreground Mask", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
            cv2.putText(detected_small, "Detection", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

            info_panel = np.zeros((display_height, display_width, 3), dtype=np.uint8)

            cv2.putText(info_panel, "TRAFFIC ANALYSIS", (20, 45),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
            cv2.putText(info_panel, "Video FPS: {:.2f}".format(video_fps), (20, 90),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(info_panel, "Processing FPS: {:.2f}".format(processing_fps), (20, 125),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(info_panel, "Processing Time: {:.2f} ms".format(processing_time), (20, 160),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(info_panel, "Frame: {}".format(frame_number), (20, 195),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(info_panel, "Moving Objects: {}".format(object_count), (20, 230),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            cv2.putText(info_panel, "Press Q to stop", (20, 270),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

            top_row = np.hstack((original_small, mask_small))
            bottom_row = np.hstack((detected_small, info_panel))
            combined = np.vstack((top_row, bottom_row))

            cv2.imshow("Day 04 - Traffic Video Analysis", combined)

            key = cv2.waitKey(1) & 0xFF
            if key == ord("q") or key == 27:
                break

    # ========================================================
    # END
    # ========================================================

    cap.release()
    out.release()
    csv_file.close()

    if SHOW_PREVIEW:
        cv2.destroyAllWindows()

    # --------------------------------------------------------
    # Calculate Statistics
    # --------------------------------------------------------

    total_frames = len(object_counts)

    average_object_count = (sum(object_counts) / total_frames) if total_frames > 0 else 0
    average_processing_time = (sum(processing_times) / total_frames) if total_frames > 0 else 0
    average_processing_fps = (sum(processing_fps_values) / total_frames) if total_frames > 0 else 0

    if min_objects is None:
        min_objects = 0

    # --------------------------------------------------------
    # Print Results
    # --------------------------------------------------------

    print("\nResults for:", video_name)
    print("--------------------------------------")
    print("Video FPS:", round(video_fps, 2))
    print("Processing FPS:", round(average_processing_fps, 2))
    print("Average Object Count:", round(average_object_count, 2))
    print("Maximum Object Count:", max_objects)
    print("Minimum Object Count:", min_objects)
    print("Average Processing Time:", round(average_processing_time, 2), "ms")
    print("Output Video:", output_path)
    print("CSV:", csv_path)

    if total_frames == 0:
        print("Warning: no frames were processed, skipping graphs.")
        return {
            "video_fps": video_fps,
            "processing_fps": average_processing_fps,
            "average_objects": average_object_count,
            "max_objects": max_objects,
            "min_objects": min_objects,
            "average_time": average_processing_time
        }

    # ========================================================
    # GRAPH 1 - OBJECT COUNT VS TIME
    # ========================================================

    plt.figure(figsize=(10, 5))
    plt.plot(time_values, object_counts)
    plt.xlabel("Time (seconds)")
    plt.ylabel("Object Count")
    plt.title(video_name + " - Object Count vs Time")
    plt.grid(True)

    graph_path = os.path.join(graphs_folder, video_name + "_object_count.png")
    plt.savefig(graph_path, dpi=150, bbox_inches="tight")
    plt.close()

    # ========================================================
    # GRAPH 2 - PROCESSING FPS (raw + smoothed)
    # ========================================================

    smoothed_fps = moving_average(processing_fps_values, FPS_SMOOTHING_WINDOW)

    plt.figure(figsize=(10, 5))
    plt.plot(time_values, processing_fps_values, alpha=0.35, label="Raw")
    plt.plot(time_values, smoothed_fps, linewidth=2, label="Smoothed (window={})".format(FPS_SMOOTHING_WINDOW))
    plt.xlabel("Time (seconds)")
    plt.ylabel("Processing FPS")
    plt.title(video_name + " - Processing FPS")
    plt.legend()
    plt.grid(True)

    fps_graph_path = os.path.join(graphs_folder, video_name + "_processing_fps.png")
    plt.savefig(fps_graph_path, dpi=150, bbox_inches="tight")
    plt.close()

    return {
        "video_fps": video_fps,
        "processing_fps": average_processing_fps,
        "average_objects": average_object_count,
        "max_objects": max_objects,
        "min_objects": min_objects,
        "average_time": average_processing_time
    }


# ============================================================
# 7 - RUN DHAKA
# ============================================================

dhaka_results = process_video(dhaka_video, "dhaka")


# ============================================================
# 8 - RUN BRITISH HIGHWAY
# ============================================================

british_results = process_video(british_video, "british_highway")


# ============================================================
# 9 - FINAL COMPARISON
# ============================================================

if dhaka_results is not None and british_results is not None:

    print("\n")
    print("==============================================")
    print("GENERALIZATION EXPERIMENT")
    print("==============================================")
    print("Note: same MOG2 parameters (history, varThreshold) were")
    print("used for both videos. Traffic density/speed differ between")
    print("the two scenes, so this is a same-settings comparison,")
    print("not a fully tuned per-scene comparison.")

    print("\nMeasurement".ljust(30), "Dhaka".ljust(15), "British Highway")
    print("-" * 65)

    print("Video FPS".ljust(30),
          "{:.2f}".format(dhaka_results["video_fps"]).ljust(15),
          "{:.2f}".format(british_results["video_fps"]))

    print("Processing FPS".ljust(30),
          "{:.2f}".format(dhaka_results["processing_fps"]).ljust(15),
          "{:.2f}".format(british_results["processing_fps"]))

    print("Average Object Count".ljust(30),
          "{:.2f}".format(dhaka_results["average_objects"]).ljust(15),
          "{:.2f}".format(british_results["average_objects"]))

    print("Maximum Object Count".ljust(30),
          str(dhaka_results["max_objects"]).ljust(15),
          str(british_results["max_objects"]))

    print("Minimum Object Count".ljust(30),
          str(dhaka_results["min_objects"]).ljust(15),
          str(british_results["min_objects"]))

    print("Average Processing Time".ljust(30),
          "{:.2f} ms".format(dhaka_results["average_time"]).ljust(15),
          "{:.2f} ms".format(british_results["average_time"]))

    # ========================================================
    # COMPARISON GRAPH
    # ========================================================

    names = ["Dhaka", "British Highway"]
    fps_values = [dhaka_results["processing_fps"], british_results["processing_fps"]]

    plt.figure(figsize=(8, 5))
    plt.bar(names, fps_values)
    plt.ylabel("Processing FPS")
    plt.title("Processing FPS Comparison")
    plt.grid(axis="y")

    comparison_graph = os.path.join(graphs_folder, "processing_fps_comparison.png")
    plt.savefig(comparison_graph, dpi=150, bbox_inches="tight")
    plt.close()

print("\n======================================")
print("PROJECT FINISHED")
print("======================================")
print("Results saved in:")
print(results_folder)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:\\Users\\PC-LAB1\\Desktop\\New folder\\test_vedio\\VID00051.mp4\\results'

## 7. حدود الطريقة (مهم)
- **افتراض المستوى الأرضي:** الـ Homography دقيقة عندما يغلب على الخلفية سطح شبه مستوٍ. المباني والأشجار القريبة تسبب parallax، و RANSAC يقلل أثرها ولا يلغيه.
- **الفيديو المشوّش (مثل بث FPV التناظري) والدوران السريع:** قد تقل الميزات، فتنخفض `camera-fit inliers` وتزيد `failures`. راقب هذين الرقمين، وتوقّع انحرافًا يتراكم في المقاطع الطويلة.
- **المنظور المائل:** البكسل الواحد لا يمثل مسافة ثابتة (القريب يبدو أكبر). لذلك السرعة بـ px/s مؤشر نسبي، ولا تُحوَّل إلى كم/س إلا مع معايرة، والأفضل تصوير عمودي من الأعلى.
- **الأجسام الصغيرة:** الأشخاص من بعيد صغار جدًا. استعمل `yolo11s/m` و`IMGSZ=960` أو أعلى، وإن لزم درّب النموذج على صور جوية.
- **المرحلة التالية (مشروع 3):** تحويل الإحداثيات المستقرة إلى إحداثيات جغرافية (تحتاج ارتفاعًا وزاوية وGPS)، وربط الأحداث بتنبيهات.